# Emodia — Treinamento de Emoção Facial com RAF-DB + EfficientNetB0

Notebook para treinar um modelo de **reconhecimento de expressão facial** mais robusto que o baseline com FER-2013.

Este notebook usa:

- Dataset **RAF-DB Face Emotion Dataset** via KaggleHub.
- Modelo **EfficientNetB0**.
- Entrada em **224x224 RGB**.
- Exportação em `.keras` e `.tflite`.
- Geração de `labels.json`, `metrics.json` e matriz de confusão.

> Nota ética: este modelo detecta **expressão facial aparente**. Ele não diagnostica depressão, ansiedade ou qualquer condição clínica. No Emodia, a visão computacional deve ser usada como sinal complementar ao texto/transcrição.


## 1. Instalação e imports

In [ ]:
!pip -q install kagglehub scikit-learn matplotlib pandas

In [ ]:
import json
import os
import random
import shutil
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow:", tf.__version__)
print("GPUs disponíveis:", tf.config.list_physical_devices("GPU"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 2. Configurações

Comece com `EPOCHS_BASE = 12` e `EPOCHS_FINE_TUNING = 12`. Se o Colab permitir, aumente depois.

Se der estouro de memória, reduza `BATCH_SIZE` para 16.


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

EPOCHS_BASE = 12
EPOCHS_FINE_TUNING = 12

MODELO_BASE = "EfficientNetB0"
NOME_MODELO = "facial_emotion_efficientnetb0_rafdb"

PASTA_SAIDA = Path("/content/emodia_cv_rafdb_outputs")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

print("Pasta de saída:", PASTA_SAIDA)


## 3. Download do dataset via KaggleHub

Dataset usado:

```python
kagglehub.dataset_download("nishchalchandel/raf-db-face-emotion-dataset")
```

Este dataset costuma vir organizado em pastas de treino/teste ou variações com nomes semelhantes. O notebook tenta localizar automaticamente as pastas.


In [ ]:
dataset_path = Path(kagglehub.dataset_download("nishchalchandel/raf-db-face-emotion-dataset"))
print("Dataset baixado em:", dataset_path)

print("\nConteúdo do diretório principal:")
for item in dataset_path.iterdir():
    print("-", item)


## 4. Localizar automaticamente pastas de treino e teste

Como datasets do Kaggle às vezes mudam a estrutura, esta célula tenta encontrar uma organização válida.

Ela procura pastas com nomes como:

- `train`
- `test`
- `valid`
- `val`
- `validation`

Se não encontrar, mostra as pastas disponíveis para você ajustar manualmente.


In [ ]:
def listar_subpastas_com_imagens(raiz):
    extensoes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    resultado = []

    for pasta in Path(raiz).rglob("*"):
        if not pasta.is_dir():
            continue

        total_imagens = sum(
            1 for arquivo in pasta.rglob("*")
            if arquivo.is_file() and arquivo.suffix.lower() in extensoes
        )

        subdirs = [p for p in pasta.iterdir() if p.is_dir()]

        if total_imagens > 0 and len(subdirs) > 0:
            resultado.append((pasta, total_imagens, len(subdirs)))

    return resultado


def encontrar_pasta_por_nome(raiz, nomes):
    candidatos = []

    for pasta in Path(raiz).rglob("*"):
        if pasta.is_dir() and pasta.name.lower() in nomes:
            classes = [p for p in pasta.iterdir() if p.is_dir()]
            if len(classes) >= 2:
                candidatos.append(pasta)

    return candidatos


train_candidates = encontrar_pasta_por_nome(dataset_path, {"train", "training"})
test_candidates = encontrar_pasta_por_nome(dataset_path, {"test", "testing"})
val_candidates = encontrar_pasta_por_nome(dataset_path, {"valid", "val", "validation"})

print("Candidatos train:", train_candidates)
print("Candidatos test :", test_candidates)
print("Candidatos val  :", val_candidates)

if not train_candidates:
    print("\nPastas com imagens e subpastas encontradas:")
    for pasta, total, n_subdirs in listar_subpastas_com_imagens(dataset_path)[:30]:
        print(f"{pasta} | imagens={total} | subpastas={n_subdirs}")

    raise FileNotFoundError(
        "Não encontrei automaticamente a pasta de treino. Veja a lista acima e ajuste train_dir manualmente."
    )

train_dir = train_candidates[0]

if test_candidates:
    test_dir = test_candidates[0]
else:
    test_dir = None

if val_candidates:
    val_dir = val_candidates[0]
else:
    val_dir = None

print("\nUsando:")
print("train_dir:", train_dir)
print("val_dir  :", val_dir)
print("test_dir :", test_dir)


## 5. Normalizar nomes de classes

Algumas versões do RAF-DB usam classes como:

- `Surprise`
- `Fear`
- `Disgust`
- `Happy`
- `Sad`
- `Angry`
- `Neutral`

Outras podem usar minúsculas. Vamos padronizar internamente.


In [ ]:
def contar_imagens_por_classe(diretorio):
    linhas = []
    diretorio = Path(diretorio)

    for classe in sorted([p for p in diretorio.iterdir() if p.is_dir()]):
        total = len([
            arquivo for arquivo in classe.rglob("*")
            if arquivo.is_file() and arquivo.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        ])

        linhas.append({
            "classe": classe.name,
            "total": total
        })

    return pd.DataFrame(linhas)


df_train_count = contar_imagens_por_classe(train_dir)
display(df_train_count)

plt.figure(figsize=(9, 4))
plt.bar(df_train_count["classe"], df_train_count["total"])
plt.title("Distribuição de imagens por classe — treino")
plt.xlabel("Classe")
plt.ylabel("Total")
plt.xticks(rotation=30)
plt.show()


## 6. Carregamento dos dados

Se existir pasta de validação separada, usamos ela. Se não existir, separamos 15% do treino para validação.

Se existir pasta de teste, usamos ela para avaliação final.


In [ ]:
if val_dir is not None:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        labels="inferred",
        label_mode="int",
        seed=SEED,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        color_mode="rgb",
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir,
        labels="inferred",
        label_mode="int",
        seed=SEED,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        color_mode="rgb",
    )
else:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        labels="inferred",
        label_mode="int",
        validation_split=0.15,
        subset="training",
        seed=SEED,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        color_mode="rgb",
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        labels="inferred",
        label_mode="int",
        validation_split=0.15,
        subset="validation",
        seed=SEED,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        color_mode="rgb",
    )

if test_dir is not None:
    test_ds = tf.keras.utils.image_dataset_from_directory(
        test_dir,
        labels="inferred",
        label_mode="int",
        shuffle=False,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        color_mode="rgb",
    )
else:
    test_ds = val_ds
    print("Aviso: nenhuma pasta test encontrada. Usando validação como teste provisório.")

class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes:", class_names)
print("Total de classes:", num_classes)


## 7. Pipeline com prefetch

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


## 8. Visualização rápida

In [ ]:
plt.figure(figsize=(10, 10))

for imagens, rotulos in train_ds.take(1):
    for i in range(min(16, len(imagens))):
        ax = plt.subplot(4, 4, i + 1)
        plt.imshow(imagens[i].numpy().astype("uint8"))
        plt.title(class_names[int(rotulos[i])])
        plt.axis("off")

plt.tight_layout()
plt.show()


## 9. Pesos por classe

Se o dataset estiver muito desbalanceado, usamos `class_weight`.


In [ ]:
labels_treino = []

for indice, classe in enumerate(class_names):
    total = len([
        arquivo for arquivo in (Path(train_dir) / classe).rglob("*")
        if arquivo.is_file() and arquivo.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    ])

    labels_treino.extend([indice] * total)

pesos = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=np.array(labels_treino),
)

class_weight = {indice: float(peso) for indice, peso in enumerate(pesos)}

print("Pesos por classe:")
for indice, peso in class_weight.items():
    print(class_names[indice], "=>", round(peso, 3))


## 10. Modelo EfficientNetB0

Usamos `include_top=False` e camadas finais próprias.

A EfficientNet do Keras já inclui normalização adequada quando usada com `preprocess_input`.


In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.06),
        tf.keras.layers.RandomZoom(0.10),
        tf.keras.layers.RandomTranslation(0.06, 0.06),
        tf.keras.layers.RandomContrast(0.10),
    ],
    name="data_augmentation",
)

base_model = tf.keras.applications.EfficientNetB0(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="face_image")

x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.35)(x)
x = tf.keras.layers.Dense(
    256,
    activation="relu",
    kernel_regularizer=tf.keras.regularizers.l2(1e-4),
)(x)
x = tf.keras.layers.Dropout(0.30)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="emotion")(x)

model = tf.keras.Model(inputs, outputs, name=NOME_MODELO)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

model.summary()


## 11. Callbacks

In [ ]:
checkpoint_path = PASTA_SAIDA / f"{NOME_MODELO}_best.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
]


## 12. Treinamento base

Primeiro treinamos apenas o classificador final.


In [ ]:
history_base = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_BASE,
    class_weight=class_weight,
    callbacks=callbacks,
)


## 13. Fine-tuning

Agora liberamos parte das últimas camadas da EfficientNetB0.

Se der overfitting, reduza as épocas ou mantenha mais camadas congeladas.


In [ ]:
base_model.trainable = True

fine_tune_at = int(len(base_model.layers) * 0.75)

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Mantém BatchNorm congelado para estabilidade
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE_TUNING,
    class_weight=class_weight,
    callbacks=callbacks,
)


## 14. Curvas de treinamento

In [ ]:
def juntar_historicos(*historicos):
    resultado = {}

    for hist in historicos:
        for chave, valores in hist.history.items():
            resultado.setdefault(chave, [])
            resultado[chave].extend(valores)

    return resultado

historico = juntar_historicos(history_base, history_fine)

plt.figure(figsize=(8, 4))
plt.plot(historico["accuracy"], label="Treino")
plt.plot(historico["val_accuracy"], label="Validação")
plt.title("Acurácia durante o treinamento")
plt.xlabel("Época")
plt.ylabel("Acurácia")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(historico["loss"], label="Treino")
plt.plot(historico["val_loss"], label="Validação")
plt.title("Loss durante o treinamento")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


## 15. Avaliação

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)

print("Test loss    :", test_loss)
print("Test accuracy:", test_accuracy)


In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
y_prob = model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)

relatorio = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0,
))


## 16. Matriz de confusão

In [ ]:
matriz = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(9, 7))
plt.imshow(matriz)
plt.title("Matriz de confusão — RAF-DB")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.xticks(np.arange(num_classes), class_names, rotation=45, ha="right")
plt.yticks(np.arange(num_classes), class_names)
plt.colorbar()

for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, matriz[i, j], ha="center", va="center")

plt.tight_layout()

confusion_path = PASTA_SAIDA / "confusion_matrix_rafdb.png"
plt.savefig(confusion_path, dpi=160, bbox_inches="tight")
plt.show()

print("Matriz salva em:", confusion_path)


## 17. Mapeamento para o Emodia

Esse mapeamento deve ser usado com cuidado. A classe facial é apenas um sinal complementar.

Ansiedade não deve ser inferida diretamente por expressão facial.


In [ ]:
def normalizar_nome_classe(nome):
    return nome.strip().lower().replace(" ", "_").replace("-", "_")

emodia_map = {}

for classe in class_names:
    classe_norm = normalizar_nome_classe(classe)

    if "happy" in classe_norm or "happiness" in classe_norm:
        emodia_map[classe] = "ALEGRIA"
    elif "sad" in classe_norm or "sadness" in classe_norm:
        emodia_map[classe] = "TRISTEZA"
    elif "angry" in classe_norm or "anger" in classe_norm:
        emodia_map[classe] = "RAIVA"
    elif "fear" in classe_norm:
        emodia_map[classe] = "MEDO"
    elif "disgust" in classe_norm:
        emodia_map[classe] = "NOJO"
    elif "neutral" in classe_norm:
        emodia_map[classe] = "NEUTRAL"
    elif "surprise" in classe_norm:
        emodia_map[classe] = "SURPRISE"
    else:
        emodia_map[classe] = "UNKNOWN"

labels = {
    "class_names": class_names,
    "class_to_index": {classe: indice for indice, classe in enumerate(class_names)},
    "facial_emotion_to_emodia": emodia_map,
    "notes": [
        "ANSIEDADE não deve ser inferida diretamente por expressão facial.",
        "Use ansiedade principalmente pela análise textual/NLP da transcrição.",
        "Este modelo detecta expressão facial aparente, não diagnóstico clínico.",
        "A visão computacional deve ser usada como sinal complementar no Emodia."
    ],
    "image_size": IMG_SIZE,
    "model": NOME_MODELO,
}

labels_path = PASTA_SAIDA / "labels.json"

with open(labels_path, "w", encoding="utf-8") as arquivo:
    json.dump(labels, arquivo, ensure_ascii=False, indent=2)

print(json.dumps(labels, ensure_ascii=False, indent=2))
print("Labels salvos em:", labels_path)


## 18. Salvar métricas

In [ ]:
metrics = {
    "dataset": "RAF-DB Face Emotion Dataset",
    "dataset_kaggle": "nishchalchandel/raf-db-face-emotion-dataset",
    "model": NOME_MODELO,
    "base_model": MODELO_BASE,
    "image_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "classification_report": relatorio,
    "classes": class_names,
    "ethical_note": (
        "Modelo de expressão facial aparente. Não deve ser usado para diagnóstico "
        "de depressão, ansiedade ou qualquer condição clínica."
    ),
}

metrics_path = PASTA_SAIDA / "metrics.json"

with open(metrics_path, "w", encoding="utf-8") as arquivo:
    json.dump(metrics, arquivo, ensure_ascii=False, indent=2)

print("Métricas salvas em:", metrics_path)


## 19. Salvar modelo `.keras`

In [ ]:
keras_model_path = PASTA_SAIDA / f"{NOME_MODELO}.keras"
model.save(keras_model_path)

print("Modelo Keras salvo em:", keras_model_path)


## 20. Exportar `.tflite`

Esse modelo pode ser usado futuramente para inferência mais leve.


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

tflite_model_path = PASTA_SAIDA / f"{NOME_MODELO}.tflite"

with open(tflite_model_path, "wb") as arquivo:
    arquivo.write(tflite_model)

print("Modelo TFLite salvo em:", tflite_model_path)
print("Tamanho MB:", round(tflite_model_path.stat().st_size / (1024 * 1024), 2))


## 21. Teste visual de predições

In [ ]:
plt.figure(figsize=(12, 12))

for imagens, rotulos in test_ds.take(1):
    probs = model.predict(imagens)
    preds = np.argmax(probs, axis=1)

    for i in range(min(16, len(imagens))):
        ax = plt.subplot(4, 4, i + 1)
        plt.imshow(imagens[i].numpy().astype("uint8"))

        real = class_names[int(rotulos[i])]
        pred = class_names[int(preds[i])]
        conf = float(np.max(probs[i]))

        titulo = f"real: {real}\npred: {pred} ({conf:.2f})"
        plt.title(titulo, fontsize=9)
        plt.axis("off")

plt.tight_layout()
plt.show()


## 22. Compactar arquivos finais para download

In [ ]:
zip_path = Path("/content/emodia_cv_rafdb_outputs.zip")

if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(
    base_name=str(zip_path).replace(".zip", ""),
    format="zip",
    root_dir=PASTA_SAIDA,
)

print("ZIP gerado em:", zip_path)
print("Arquivos incluídos:")
for arquivo in sorted(PASTA_SAIDA.iterdir()):
    print("-", arquivo.name)


## 23. Download no Colab

In [ ]:
from google.colab import files

files.download("/content/emodia_cv_rafdb_outputs.zip")


## 24. Onde colocar no repositório

Depois de baixar o ZIP:

```bash
mkdir -p ml/notebooks/cv ml/models/cv ml/reports/cv

# Notebook:
ml/notebooks/cv/treino_emocao_facial_rafdb_efficientnetb0_colab.ipynb

# Modelos:
ml/models/cv/facial_emotion_efficientnetb0_rafdb.keras
ml/models/cv/facial_emotion_efficientnetb0_rafdb.tflite
ml/models/cv/labels.json

# Relatórios:
ml/reports/cv/confusion_matrix_rafdb.png
ml/reports/cv/metrics.json
```

Commit sugerido:

```bash
git add ml
git commit -m "feat: add RAF-DB facial emotion training notebook"
```
